In [0]:
from pyspark.sql.functions import *

### Bronze Layer - Raw Data Ingestion

### Reading CSV Files

In [0]:
customers_df=spark.read.format('csv').option("header",True).option("inferSchema",True).load("/Volumes/e-commerce_databricks_project/default/e-commerce_datasets/olist_customers_dataset.csv")
order_items_df=spark.read.format('csv').option("header",True).option("inferSchema",True).load("/Volumes/e-commerce_databricks_project/default/e-commerce_datasets/olist_order_items_dataset.csv")
order_payments_df=spark.read.format('csv').option("header",True).option("inferSchema",True).load("/Volumes/e-commerce_databricks_project/default/e-commerce_datasets/olist_order_payments_dataset.csv")
orders_df=spark.read.format('csv').option("header",True).option("inferSchema",True).load("/Volumes/e-commerce_databricks_project/default/e-commerce_datasets/olist_orders_dataset.csv")
products_df=spark.read.format('csv').option("header",True).option("inferSchema",True).load("/Volumes/e-commerce_databricks_project/default/e-commerce_datasets/olist_products_dataset.csv")

### Checking schema and displaying data

In [0]:
customers_df.show()
order_items_df.show()
order_payments_df.show()
orders_df.show()
products_df.show()

In [0]:
customers_df.printSchema()
order_items_df.printSchema()
order_payments_df.printSchema()
orders_df.printSchema()
products_df.printSchema()

### Save as Bronze (Delta Table)

In [0]:
customers_df.write.format('delta').mode('overwrite').save('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/customers')
order_items_df.write.format('delta').mode('overwrite').save('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/order_items')
order_payments_df.write.format('delta').mode('overwrite').save('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/order_payments')
orders_df.write.format('delta').mode('overwrite').save('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/orders')
products_df.write.format('delta').mode('overwrite').save('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/products')

### Bronze Data Validation

### Load All The Tables

In [0]:
customers=spark.read.format('delta').load('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/customers')
order_items=spark.read.format('delta').load('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/order_items')
payments=spark.read.format('delta').load('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/order_payments')
orders=spark.read.format('delta').load('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/orders')
products=spark.read.format('delta').load('/Volumes/e-commerce_databricks_project/default/bronze_deltatable'+'/products')

### Creating Validation Function

In [0]:
from pyspark.sql.functions import col, count
def validate_data(df, table_name):
    print(f"Validating Table: {table_name}")
    # Row count
    print("Total Rows:", df.count())
    # Null check
    print("\nNull Values:")
    df.select([count(when(col(c).isNull(),c)).alias(c + "_nulls") for c in df.columns]).display()
    # Duplicate check
    dup_count = df.count() - df.dropDuplicates().count()
    print("\nDuplicate Rows:", dup_count)
    # Show sample data
    print("\nSample Data:")
    df.show(5)

In [0]:
validate_data(customers,"customers")
validate_data(order_items,"order_items")
validate_data(payments,"payments")
validate_data(orders,"orders")
validate_data( products,"products")

### Checking Negative Values

In [0]:
payments.filter(col("payment_value") <= 0).show()

In [0]:
orders.display()

### Silver Layer (Data Cleaning And Transformation)

### Load Bronze Tables

In [0]:
customers = spark.read.format("delta").load("/Volumes/e-commerce_databricks_project/default/bronze_deltatable/customers")
order_items = spark.read.format("delta").load("/Volumes/e-commerce_databricks_project/default/bronze_deltatable/order_items")
payments = spark.read.format("delta").load("/Volumes/e-commerce_databricks_project/default/bronze_deltatable/order_payments")
orders = spark.read.format("delta").load("/Volumes/e-commerce_databricks_project/default/bronze_deltatable/orders")
products = spark.read.format("delta").load("/Volumes/e-commerce_databricks_project/default/bronze_deltatable/products")

### Data Cleaning

In [0]:
customers_clean = customers.dropDuplicates()

### Handle Null in Orders

In [0]:
from pyspark.sql.functions import col
orders_clean = orders.withColumn("order_approved_at",col("order_approved_at").cast("timestamp")).fillna({"order_approved_at": "1970-01-01 00:00:00"}).dropDuplicates()

### Removing Invalid Values From Payments

In [0]:
payments_clean = payments.filter(col("payment_value") > 0).dropDuplicates()

### Handling Nulls in Products

In [0]:
products_clean = products.fillna({"product_category_name": "unknown","product_name_lenght": 0,"product_description_lenght": 0,"product_photos_qty": 0}).dropDuplicates()

In [0]:
order_items_clean = order_items.dropDuplicates()

### JOINS

In [0]:
ecommerce_silver = orders_clean.alias("o") \
    .join(customers_clean.alias("c"), "customer_id", "left") \
    .join(order_items_clean.alias("oi"), "order_id", "left") \
    .join(products_clean.alias("p"), "product_id", "left") \
    .join(payments_clean.alias("pay"), "order_id", "left")

In [0]:
ecommerce_silver.display()

### Add Business Column

In [0]:
from pyspark.sql.functions import year, month
ecommerce_silver = ecommerce_silver \
    .withColumn("order_year", year("order_purchase_timestamp")) \
    .withColumn("order_month", month("order_purchase_timestamp")) \
    .withColumn("total_price", col("price") + col("freight_value"))

### Save SILVER Delta Tables

In [0]:
ecommerce_silver.write.format("delta") \
    .mode("overwrite") \
    .save("/Volumes/e-commerce_databricks_project/default/silver_deltatable")

### GOLD LAYER (Analytics + KPIs)

### Top Customers

In [0]:
from pyspark.sql.functions import sum
top_customers = ecommerce_silver.groupBy("customer_id").agg(sum("payment_value").alias("total_spent")).orderBy(col("total_spent").desc())

### Monthly Revenue

In [0]:
monthly_revenue = ecommerce_silver.groupBy("order_year", "order_month").agg(sum("payment_value").alias("revenue")).orderBy("order_year", "order_month")
monthly_revenue.display()

### Best Selling Products

In [0]:
from pyspark.sql.functions import count
top_products = ecommerce_silver.groupBy("product_id", "product_category_name").agg(count("order_item_id").alias("total_orders")).orderBy(col("total_orders").desc())

### Repeat Customers

In [0]:
repeat_customers = ecommerce_silver.groupBy("customer_id").agg(count("order_id").alias("order_count")).filter(col("order_count") > 1)

### Customer Segmentation

In [0]:
from pyspark.sql.functions import when, sum, col

customer_segments = ecommerce_silver.groupBy("customer_id") \
    .agg(sum("payment_value").alias("total_spent")) \
    .withColumn("segment",
        when(col("total_spent") > 5000, "High Value")
        .when(col("total_spent") > 2000, "Medium Value")
        .otherwise("Low Value")
    )

### ORDER DELIVERY PERFORMANCE

In [0]:
from pyspark.sql.functions import datediff

delivery_performance = ecommerce_silver \
    .withColumn("delivery_delay",
        datediff("order_delivered_customer_date",
                 "order_estimated_delivery_date")
    )

### TOP CITIES BY REVENUE

In [0]:
from pyspark.sql.functions import sum

top_cities = ecommerce_silver.groupBy("customer_city") \
    .agg(sum("payment_value").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())

### PAYMENT METHOD ANALYSIS

In [0]:
payment_analysis = ecommerce_silver.groupBy("payment_type") \
    .agg(sum("payment_value").alias("total_amount"))

### PRODUCT CATEGORY PERFORMANCE

In [0]:
category_performance = ecommerce_silver.groupBy("product_category_name") \
    .agg(
        sum("payment_value").alias("revenue"),
        count("order_item_id").alias("orders")
    ) \
    .orderBy(col("revenue").desc())

### Top product per category

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("product_category_name") \
    .orderBy(col("price").desc())

top_product_per_category = ecommerce_silver \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1)

### Saving Gold Tables

In [0]:
gold_path='/Volumes/e-commerce_databricks_project/default/gold_deltatable/'
top_customers.write.format("delta").mode("overwrite").save(gold_path+"top_customers")
monthly_revenue.write.format("delta").mode("overwrite").save(gold_path+"monthly_revenue")
top_products.write.format("delta").mode("overwrite").save(gold_path+"top_products")
repeat_customers.write.format("delta").mode("overwrite").save(gold_path+"repeat_customers")
customer_segments.write.format("delta").mode("overwrite").save(gold_path+"customer_segments")
delivery_performance.write.format("delta").mode("overwrite").save(gold_path+"delivery_performance")
top_cities.write.format("delta").mode("overwrite").save(gold_path+"top_cities")
payment_analysis.write.format("delta").mode("overwrite").save(gold_path+"payment_analysis")
category_performance.write.format("delta").mode("overwrite").save(gold_path+"category_performance")
top_product_per_category.write.format("delta").mode("overwrite").save(gold_path+"top_product_per_category")